# 03 · Data Quality & Cleaning

The raw `fact_sales` CSV is deliberately messy: nulls, duplicate
transactions, non-numeric quantities, negative quantities, out-of-range
discounts, and orphan product foreign keys. The dimension text (category,
region) also has inconsistent casing/whitespace.

Your job: audit and clean it into a trustworthy dataset.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "utils" / "spark.py").exists() and _root != _root.parent:
    _root = _root.parent
for _p in (str(_root), str(_root / "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from utils.spark import build_session, read_dim, read_sales_raw, read_sales_typed

spark = build_session("nb03")
print("Spark", spark.version, "ready")

sales_raw = read_sales_raw(spark)
products = read_dim(spark, "dim_products")
stores = read_dim(spark, "dim_stores")
print("raw sales rows:", sales_raw.count())
sales_raw.show(5, truncate=False)

## Task 1: Audit the raw data

Compute three integers on `sales_raw`:
- `n_null_customer`: rows where `customer_id` is null
- `n_bad_qty`: rows where `quantity` is **not** a valid integer (hint: `try_cast(... as int)` is null)
- `n_dupes`: duplicate transaction rows (total rows − distinct `txn_id`)

In [ ]:
n_null_customer = sales_raw.filter(F.col("customer_id").isNull()).count()
n_bad_qty = sales_raw.filter(F.expr("try_cast(quantity as int)").isNull()).count()
n_dupes = sales_raw.count() - sales_raw.select("txn_id").distinct().count()
print(n_null_customer, n_bad_qty, n_dupes)

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert n_null_customer == sales_raw.filter(F.col("customer_id").isNull()).count()
assert n_bad_qty > 0, "expected some non-numeric quantities"
assert n_dupes > 0, "expected duplicate transactions"
print("✅ Task 1 passed")

## Task 2: Deduplicate transactions

Produce `deduped` so each `txn_id` appears exactly once.

In [ ]:
deduped = sales_raw.dropDuplicates(["txn_id"])
print("rows:", deduped.count())

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert deduped.count() == deduped.select("txn_id").distinct().count()
assert deduped.count() == sales_raw.select("txn_id").distinct().count()
print("✅ Task 2 passed — deduped rows:", deduped.count())

## Task 3: Safely impose types

From `deduped`, build `typed` with `quantity`→int, `unit_price`→double, `discount`→double, `txn_date`→date, using **safe** casts (bad quantities become null).

In [ ]:
typed = (
    deduped
    .withColumn("txn_date", F.to_date("txn_date"))
    .withColumn("quantity", F.expr("try_cast(quantity as int)"))
    .withColumn("unit_price", F.expr("try_cast(unit_price as double)"))
    .withColumn("discount", F.expr("try_cast(discount as double)"))
)
typed.printSchema()

In [ ]:
# ✅ CHECK — run this cell to grade your answer
_t = dict(typed.dtypes)
assert _t["quantity"] == "int" and _t["discount"] == "double" and _t["txn_date"] == "date"
assert typed.filter(F.col("quantity").isNull()).count() > 0
print("✅ Task 3 passed")

## Task 4: Normalize categorical text

`products.category` has inconsistent casing/whitespace (`'  beverages '`, `'BEVERAGES'`, …). Build `products_clean` with a normalized `category` (trim + title-case) so each real category collapses to a single value.

In [ ]:
products_clean = products.withColumn("category", F.initcap(F.trim(F.col("category"))))
_before = products.select("category").distinct().count()
_after = products_clean.select("category").distinct().count()
print("distinct categories before:", _before, "after:", _after)

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert products_clean.select("category").distinct().count() < products.select("category").distinct().count()
assert products_clean.select("category").distinct().count() == 6
print("✅ Task 4 passed")

## Task 5: Find orphan foreign keys

Find sales whose `product_id` has no match in `products` (orphan FKs). Produce `orphans` (distinct `product_id`) and the integer `n_orphan_products`. Use a `left_anti` join from `typed`.

In [ ]:
orphans = typed.join(products, "product_id", "left_anti").select("product_id").distinct()
n_orphan_products = orphans.count()
print("orphan product_ids:", n_orphan_products)
orphans.show(5)

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert n_orphan_products > 0
assert orphans.join(products, "product_id", "inner").count() == 0
print("✅ Task 5 passed — orphans:", n_orphan_products)

## Task 6: Assemble the clean dataset

Build the final `clean_sales`: from `typed`, drop rows with non-positive/null quantity and discounts outside [0, 1], inner-join to `products_clean` (dropping orphans and attaching `category`, `brand`), and add a `revenue` column.

In [ ]:
clean_sales = (
    typed
    .filter((F.col("quantity") > 0) & F.col("discount").between(0, 1))
    .join(products_clean.select("product_id", "category", "brand"), "product_id")
    .withColumn(
        "revenue",
        F.col("quantity") * F.col("unit_price") * (1 - F.col("discount")),
    )
)
clean_sales.select("txn_id", "category", "quantity", "discount", "revenue").show(5)

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert clean_sales.filter(~F.col("discount").between(0, 1)).count() == 0
assert clean_sales.filter(F.col("quantity") <= 0).count() == 0
assert clean_sales.join(products, "product_id", "left_anti").count() == 0
assert {"revenue", "category"} <= set(clean_sales.columns)
print("✅ Task 6 passed — clean rows:", clean_sales.count())

---
🎉 **Solution notebook** — all cells should run top to bottom.